# Fan-out latency — example analysis

## Naive direct SHM

One run of the harness: 200k messages at 250k msg/s, mixed types, 1024 slots. Run from the repository root. The Make target owns and bounds the producer lifetime and removes its SHM segment after collection.

```bash
make run-direct-test DIRECT_PINNED=0 DIRECT_SHM=/fanout_demo \
    SHM_SLOTS=1024 MESSAGE_COUNT=200000 MESSAGE_RATE=250000 \
    LATENCY_CSV="$PWD/data/latency-naive.csv"
gzip -9 -n -f data/latency-naive.csv
```

In [ ]:
import numpy as np
import pandas as pd
from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter
from bokeh.plotting import figure

output_notebook()

BLUE = "#2a78d6"

naive_data = pd.read_csv("data/latency-naive.csv.gz")
ns = naive_data["latency_ns"].to_numpy()
ns.size, ns.min(), int(ns.mean()), ns.max()

## Distribution

Clipped at p99.9 for display purposes

In [ ]:
clip = np.percentile(ns, 99.9)
counts, edges = np.histogram(ns[ns <= clip], bins=60)
src = ColumnDataSource(dict(count=counts, left=edges[:-1], right=edges[1:],
                            mid=(edges[:-1] + edges[1:]) / 2))

p = figure(height=320, sizing_mode="stretch_width",
           title=f"Delivery latency, clipped at p99.9 = {clip:,.0f} ns",
           x_axis_label="latency (ns)", y_axis_label="messages",
           tools="pan,box_zoom,reset,save")
p.quad(source=src, bottom=0, top="count", left="left", right="right",
       fill_color=BLUE, line_color="white")
p.y_range.start = 0
p.xaxis.formatter = p.yaxis.formatter = NumeralTickFormatter(format="0,0")
p.add_tools(HoverTool(tooltips=[("latency", "@mid{0,0} ns"),
                                ("messages", "@count{0,0}")], mode="vline"))
show(p)

## Tail

Log axis on both scales — a mean hides the tail entirely.

In [ ]:
# Plotted against "distance from 100th percentile" on a log axis, so the tail gets
# real width instead of being squeezed into the last few pixels.
# Stops at p99.99: beyond that is a handful of samples out of 200k, not a stable figure.
pcts = np.array([50, 75, 90, 99, 99.9, 99.99])
curve = ColumnDataSource(dict(x=100 - pcts, ns=np.percentile(ns, pcts),
                              pct=[f"p{v:g}" for v in pcts]))

q = figure(height=320, sizing_mode="stretch_width",
           x_axis_type="log", y_axis_type="log",
           x_range=(70, 0.0025),  # reversed: p50 on the left, p99.99 on the right
           title="Latency tail",
           x_axis_label="percentile", y_axis_label="latency (ns, log)",
           tools="pan,box_zoom,reset,save")
q.line(source=curve, x="x", y="ns", color=BLUE, width=2)
q.scatter(source=curve, x="x", y="ns", color=BLUE, size=9,
          line_color="white", line_width=2)
q.xaxis.ticker = list(100 - pcts)
q.xaxis.major_label_overrides = {100 - v: f"p{v:g}" for v in pcts}
q.yaxis.ticker = [100, 200, 500, 1000, 2000, 5000]
q.yaxis.formatter = NumeralTickFormatter(format="0,0")
q.add_tools(HoverTool(tooltips=[("percentile", "@pct"), ("latency", "@ns{0,0} ns")]))
show(q)

pd.DataFrame([{f"p{v:g}": int(x) for v, x in zip(pcts, np.percentile(ns, pcts))}])

## Pinned direct SHM

Repeat the same workload with the producer pinned to isolated CPU 4 and the consumer pinned to isolated CPU 7. This laptop was booted with CPUs 4–7 isolated and SMT disabled; KDE's performance profile maps to `intel_pstate` with EPP set to `performance`. Run from the repository root:

```bash
make run-direct-test DIRECT_PINNED=1 DIRECT_SHM=/fanout_pinned \
    SHM_SLOTS=1024 MESSAGE_COUNT=200000 MESSAGE_RATE=250000 \
    LATENCY_CSV="$PWD/data/latency-pinned.csv"
gzip -9 -n -f data/latency-pinned.csv
```

## Pinned UDP over veth

The third run inserts the current blocking-socket transport between the two SHM rings. Producer, sender, receiver, and consumer are pinned to isolated CPUs 4, 5, 6, and 7 respectively. `net-up` also removes any active `netem` profile, so this measures the clean local veth path. Run from the repository root:

```bash
make net-up
make run-test MESSAGE_COUNT=200000 MESSAGE_RATE=250000 SHM_SLOTS=1024 \
    LATENCY_CSV="$PWD/data/latency-network.csv"
gzip -9 -n -f data/latency-network.csv
```

The sender joins the producer's live edge, so its sample can be slightly smaller than 200k messages; the actual size is retained in the comparison table.

## Direct SHM and UDP/veth

The first pair isolates the effect of CPU affinity. The third run adds two SHM hand-offs and the current `sendto`/`recvfrom` UDP path over a clean veth. These are single runs; repeated, interleaved runs are needed before treating small differences as stable.

In [ ]:
pinned_data = pd.read_csv("data/latency-pinned.csv.gz")
network_data = pd.read_csv("data/latency-network.csv.gz")
datasets = {
    "naive direct SHM": naive_data,
    "pinned direct SHM (CPU 4 → 7)": pinned_data,
    "UDP/veth (CPU 4 → 5 → 6 → 7)": network_data,
}
runs = {name: data["latency_ns"].to_numpy() for name, data in datasets.items()}

def describe_run(name, values):
    percentiles = np.percentile(values, [50, 99, 99.9, 99.99])
    seq = datasets[name]["seq"].to_numpy()
    expected = int(seq.max() - seq.min() + 1)
    unique = np.unique(seq).size
    return {
        "run": name,
        "samples": values.size,
        "missing seq": expected - unique,
        "duplicates": values.size - unique,
        "backward jumps": int(np.count_nonzero(np.diff(seq) < 0)),
        "mean": int(values.mean()),
        "p50": int(percentiles[0]),
        "p99": int(percentiles[1]),
        "p99.9": int(percentiles[2]),
        "p99.99": int(percentiles[3]),
        "max": int(values.max()),
    }

summary = pd.DataFrame([describe_run(name, values) for name, values in runs.items()])
display(summary.set_index("run"))

colors = {
    "naive direct SHM": "#d65f5f",
    "pinned direct SHM (CPU 4 → 7)": BLUE,
    "UDP/veth (CPU 4 → 5 → 6 → 7)": "#2ca25f",
}
comparison = figure(height=360, sizing_mode="stretch_width",
                    x_axis_type="log", y_axis_type="log",
                    x_range=(70, 0.0025),
                    title="End-to-end latency tail",
                    x_axis_label="percentile", y_axis_label="latency (ns, log)",
                    tools="pan,box_zoom,reset,save")
for name, values in runs.items():
    source = ColumnDataSource(dict(
        x=100 - pcts,
        ns=np.percentile(values, pcts),
        pct=[f"p{value:g}" for value in pcts],
        run=[name] * len(pcts),
    ))
    comparison.line(source=source, x="x", y="ns", color=colors[name],
                    width=2, legend_label=name)
    comparison.scatter(source=source, x="x", y="ns", color=colors[name],
                       size=8, line_color="white", line_width=1)
comparison.xaxis.ticker = list(100 - pcts)
comparison.xaxis.major_label_overrides = {100 - value: f"p{value:g}" for value in pcts}
comparison.yaxis.formatter = NumeralTickFormatter(format="0,0")
comparison.legend.location = "top_left"
comparison.legend.click_policy = "hide"
comparison.add_tools(HoverTool(tooltips=[("run", "@run"),
                                         ("percentile", "@pct"),
                                         ("latency", "@ns{0,0} ns")]))
show(comparison)

### Sequence integrity

The sequence columns distinguish messages absent from the observed range, duplicate IDs, and backward jumps. In the current naive run, a scheduler stall let the producer lap the consumer once, producing 247 legitimate missing IDs. It has no duplicates or backward jumps. The pinned and network samples have no gaps or ordering anomalies inside their observed ranges. The network run starts at the sender's live edge, so producer messages before its first observed ID are intentionally outside this accounting.

Before the ring fix, the naive direct-SHM sample contained 2,856 missing IDs, three duplicates, and two backward jumps. Commit `ac75063` captures a deterministic regression test: it pauses a writer after overwriting a one-slot ring's payload but before publishing the new sequence, and the old implementation accepts the new frame as the previous generation. The fixed writer invalidates the slot before modifying it; `frame_len` and payload words are atomic, and the reader validates the sequence after copying. The same test is also clean under ThreadSanitizer.

### Observation

In this run, pinning the direct SHM path changed p50 from 189 ns to 161 ns and p99 from 1,218,330 ns to 865 ns. The clean UDP/veth path measured 4,936 ns at p50, 13,043 ns at p99, 16,667 ns at p99.9, and 90,424 ns at p99.99. Relative to pinned direct SHM, the current two-ring blocking-socket transport therefore adds about 4.8 us at p50 and exposes a substantially larger far tail.

`consumer --csv` records samples in a preallocated memory buffer and writes the file after the hot loop. The results are still diagnostic single-run numbers; final measurements should compare several interleaved runs.

## Ring overwrite fix — ablation

`data/ring-fix-ablation.csv` keeps the per-run aggregates from the intermediate implementations. The series have separate names because only variants inside the same series are compared. The publication-window series isolates the extra sequence invalidation store. Its byte-array payload is still a formal C++ data race, so it is a diagnostic intermediate rather than a valid final design. The strict-fix series is an interleaved A/B comparison of the original ring and the final atomic representation. The copy-layout series explores strict atomic representations; its p50 is useful for code-generation direction, while its p99 was visibly affected by machine state and is not used to claim a layout effect. The writer-cursor series is a seven-run interleaved A/B comparison on the final layout.

The effects remain distinct:

- Invalidating the slot before overwrite closed the logical publication window at a median p50 cost of 1 ns (192 → 193 ns).
- Replacing racy byte copies with lock-free atomic words made the implementation valid in the C++ memory model. In the interleaved A/B series, the complete strict fix changed median p50 from 194 to 159 ns and p99 from 779 to 761 ns.
- At the original payload offset, using 64-bit words for the aligned middle instead of 32-bit words throughout changed median p50 from 274 to 228 ns.
- Moving the payload from offset 12 to its natural 8-byte boundary and using one uniform 64-bit loop changed median p50 from 228 to 198 ns in the exploratory layout series. This is 8-byte payload alignment, not cache-line alignment; each slot was already aligned to 64 bytes.
- Keeping the single writer's cursor locally instead of loading the shared atomic edge for every publication changed median p50 from 166 to 161 ns. Median p99 changed from 748 to 764 ns, which is within the observed run-to-run noise.

On this x86-64 build, acquire/release operations on these lock-free words compile to ordinary loads and stores: the producer and consumer hot paths contain no locked instruction or memory fence.

In [ ]:
ablation = pd.read_csv("data/ring-fix-ablation.csv")
ablation_summary = (
    ablation.groupby(["series", "variant"], sort=False)
    .agg(
        runs=("run", "count"),
        samples=("samples", "first"),
        p50_median_ns=("p50_ns", "median"),
        p50_min_ns=("p50_ns", "min"),
        p50_max_ns=("p50_ns", "max"),
        p99_median_ns=("p99_ns", "median"),
    )
    .astype(int)
)
display(ablation_summary)